In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.stats.power import TTestIndPower

In [ ]:
PILOT_CSV = r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\pilot_20260424_001142.csv"
RESULTS_DIR = Path(r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results")

In [ ]:
ENERGY_COL = "joules"
GROUP_COLS = ["language", "algorithm", "size"]
 

In [ ]:
N_LANGUAGES       = 6
N_COMPARISONS     = (N_LANGUAGES * (N_LANGUAGES - 1)) // 2   # = 15
ALPHA_NOMINAL     = 0.05
ALPHA_CORRECTED   = ALPHA_NOMINAL / N_COMPARISONS             # Bonferroni
EFFECT_SIZE_D     = 0.5                                        # medium (Cohen 1988)
POWER_TARGET      = 0.80                                       # 80% power

In [ ]:
# 1. Load pilot data

df = pd.read_csv(PILOT_CSV)
print(f"Loaded {len(df)} rows from pilot CSV.")
print(f"Columns: {list(df.columns)}\n")

In [ ]:
# 2. Compute per-cell statistics
# --------------------------------------------------
stats = df.groupby(GROUP_COLS)[ENERGY_COL].agg(
    count="count",
    mean="mean",
    std="std"
).reset_index()
stats["cv_pct"] = (stats["std"] / stats["mean"]) * 100

In [ ]:
# 3. Power analysis
    # --------------------------------------------------
    # TTestIndPower solves for sample size per group given:
    #   effect_size : Cohen's d
    #   alpha       : Bonferroni-corrected significance level
    #   power       : desired statistical power
    #   alternative : 'two-sided'
analysis = TTestIndPower()
n_required = analysis.solve_power(
    effect_size=EFFECT_SIZE_D,
    alpha=ALPHA_CORRECTED,
    power=POWER_TARGET,
    alternative="two-sided"
)
n_required_ceil = int(np.ceil(n_required))
 